In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
import zipfile
import io

class Vocabulary:
    def __init__(self, vocab, word_dict):
        self.vocab = vocab
        self.word_dict = word_dict

    def get_word_from_index(self, index):
        return self.vocab[index]

    def get_index_from_word(self, word):
        if word in self.word_dict:
          return self.word_dict[word]

        return self.word_dict["<UNK>"]

class Text8Dataset:
    def __init__(self, file_name, window_size):
        self.file_name = file_name
        self.window_size = window_size
        self.words = self.read_words()
        self.vocabulary = self.create_vocabulary()

    def create_training_data(self):
        # for each word in training data,
        # want context consisting of window_size words before and after
        # input will be context indices and output is word index
        # training example is (context_indices, word_index)

        X = []
        y = []
        # handle first and last window_size words differently (to avoid out of bounds errors)
        # for i in range(self.window_size, len(self.vocabulary.vocab) - self.window_size):
        for i, word in enumerate(self.words):
            if i >= self.window_size and i < len(self.words) - self.window_size:
                context = [self.vocabulary.get_index_from_word(self.words[i-j]) for j in range(1, self.window_size+1)]
                context += [self.vocabulary.get_index_from_word(self.words[i+j]) for j in range(1, self.window_size+1)]
                X.append(context)
                y.append(self.vocabulary.get_index_from_word(word))
        return X, y

    def read_words(self):
        print("reading words...")
        with zipfile.ZipFile(f"{self.file_name}.zip", "r") as zip_ref:
            text = zip_ref.read(self.file_name).decode("utf-8")
            words = text.split(" ")

            # first index is a blank string
            print("done reading words...")
            return words[1:]
        return []

    def create_vocabulary(self):
        words = self.words
        word_counts = {}
        for word in words:
            if word in word_counts:
                word_counts[word] += 1
            else:
                word_counts[word] = 1

        vocab = []
        word_dict = {}
        current_index = 0 # use a separate counter for frequent words

        for word, count in word_counts.items():
            if count >= 20:
              word_dict[word] = current_index
              vocab.append(word)
              current_index += 1

        # the index for UNK should be the next available index
        word_dict["<UNK>"] = current_index
        # optionally add <UNK> to vocab if needed, but not strictly necessary for indexing

        vocabulary = Vocabulary(vocab, word_dict)
        return vocabulary

text8_dataset = Text8Dataset("text8", 2)
train_x, train_y = text8_dataset.create_training_data()

reading words...
done reading words...


In [ ]:
VOCAB_SIZE = len(text8_dataset.vocabulary.word_dict.keys())
print(f"VOCAB_SIZE: {VOCAB_SIZE}")

# size of word vectors to learn
EMBEDDING_SIZE = 300

VOCAB_SIZE: 31893


In [5]:
# prepare data
from torch.utils.data import TensorDataset
X_tensor = torch.tensor(train_x, dtype=torch.long)
y_tensor = torch.tensor(train_y, dtype=torch.long)
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

In [ ]:
for X, y in dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([128, 4])
Shape of y: torch.Size([128]) torch.int64


In [ ]:
class CBOWNetwork(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        super().__init__()
        self.embedding_layer = nn.Embedding(
            vocab_size,
            embedding_size
        )
        self.linear_layer = nn.Linear(
            embedding_size,
            vocab_size
        )

    def forward(self, x):
        x = self.embedding_layer(x)
        # average
        x = x.mean(axis=1)
        x = self.linear_layer(x)
        return x

max_index_in_train_x = 0
if 'train_x' in globals() and train_x: # ensure train_x exists and is not empty
    for sublist in train_x:
        if sublist: # ensure sublist is not empty
            max_index_in_train_x = max(max_index_in_train_x, max(sublist))

max_index_in_train_y = 0
if 'train_y' in globals() and train_y: # ensure train_y exists and is not empty
    max_index_in_train_y = max(train_y)

# the vocab size should be 1 + the maximum index found in the data
# because indices are 0-based. If no data, default to 1.
adjusted_vocab_size = max(max_index_in_train_x, max_index_in_train_y) + 1

print(f"Original VOCAB_SIZE (from global scope): {VOCAB_SIZE}")
print(f"Max index in train_x: {max_index_in_train_x}")
print(f"Max index in train_y: {max_index_in_train_y}")
print(f"Adjusted VOCAB_SIZE (calculated from data): {adjusted_vocab_size}")

# use the adjusted_vocab_size for the model
model = CBOWNetwork(adjusted_vocab_size, EMBEDDING_SIZE).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Original VOCAB_SIZE (from global scope): 31893
Max index in train_x: 31892
Max index in train_y: 31892
Adjusted VOCAB_SIZE (calculated from data): 31893


In [7]:
def train(model, dataloader, loss_fn, optimizer, device):
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            print(f"loss: {loss.item():>7f}  [{batch:>5d}/{len(dataloader):>5d}]")

In [ ]:
def test(model, dataloader, loss_fn, device):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= len(dataloader.dataset)
    correct /= len(dataloader.dataset)
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
epochs = 1
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(model, dataloader, loss_fn, optimizer, device)

In [ ]:
word_index = torch.tensor([5]).to(device)
word_vector = model.embedding_layer(word_index)
print(word_vector)

In [ ]:
# 'model.embedding' is the trained layer
trained_weights = model.embedding_layer.weight.data

# Convert to numpy for external tools 
weights_numpy = trained_weights.cpu().numpy()
print(weights_numpy[5])

In [ ]:
import numpy as np

# weights is numpy array of all word vectors
# dataset is text8dataset object
def get_closest_words_from_index(dataset, weights, index, n):
    word_vector = weights[index]
    word_vector_magnitude = np.linalg.norm(word_vector)

    similarities = [] 
    for i, word in enumerate(dataset.vocabulary.vocab):
        current_vector = weights[i]
        current_magnitude = np.linalg.norm(current_vector)
        cosine_similarity = np.dot(word_vector, current_vector) / (word_vector_magnitude * current_magnitude)
        similarities.append({ "word": word, "similarity": cosine_similarity })
    similarities_sorted = sorted(similarities, key=lambda w: w["similarity"], reverse=True)
    return [w["word"] for w in similarities_sorted[0:n]]

def get_closest_words_from_vector(dataset, weights, vector, n):
    word_vector_magnitude = np.linalg.norm(vector)

    similarities = [] 
    for i, word in enumerate(dataset.vocabulary.vocab):
        current_vector = weights[i]
        current_magnitude = np.linalg.norm(current_vector)
        cosine_similarity = np.dot(vector, current_vector) / (word_vector_magnitude * current_magnitude)
        similarities.append({ "word": word, "similarity": cosine_similarity })
    similarities_sorted = sorted(similarities, key=lambda w: w["similarity"], reverse=True)
    return [w["word"] for w in similarities_sorted[0:n]]


man_index = text8_dataset.vocabulary.get_index_from_word("man")
king_index = text8_dataset.vocabulary.get_index_from_word("king")
woman_index = text8_dataset.vocabulary.get_index_from_word("woman")
queen_index = text8_dataset.vocabulary.get_index_from_word("queen")
print(man_index)
print(woman_index)
print(king_index)
print(queen_index)

man_vec = weights_numpy[man_index]
king_vec = weights_numpy[king_index]
woman_vec = weights_numpy[woman_index]

man_to_king_plus_woman = (king_vec - man_vec) + woman_vec

similarities = get_closest_words_from_index(text8_dataset, weights_numpy, queen_index, 50)
print(similarities)

similarities = get_closest_words_from_vector(text8_dataset, weights_numpy, man_to_king_plus_woman, 62)
print(similarities)
print("queen" in similarities)

In [56]:
word_index = text8_dataset.vocabulary.get_index_from_word("florida")
print(word_index)

similarities = get_closest_words_from_index(text8_dataset, weights_numpy, word_index, 50)
print(similarities)

3419
['florida', 'tijuana', 'montana', 'tennessee', 'california', 'madhya', 'miami', 'maryland', 'fort', 'indiana', 'vermont', 'ohio', 'santo', 'randall', 'croydon', 'arizona', 'michigan', 'anaheim', 'bay', 'southwest', 'downtown', 'kabaddi', 'southampton', 'swaim', 'bethlehem', 'gangsta', 'strokes', 'privatized', 'city', 'atlanta', 'columbine', 'bribe', 'connecticut', 'monroe', 'massachusetts', 'ithaca', 'alexei', 'gary', 'cedar', 'carolina', 'recombination', 'dakota', 'visit', 'reagan', 'aceh', 'shaftesbury', 'maine', 'bangalore', 'university', 'lupin']


In [ ]:
torch.save(model.state_dict(), 'model_weights.pth')